## Patient Journey - T2D
#### Data source used

* PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_DIAGNOSIS
* PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_PRODUCT 
* finerenone_intermediate.diabetes_product
* finerenone_intermediate.diabetes_product_injection_mapping

#### Steps:
1. Query TYPE 2 DIABETES diagnosis claims data after 2016-01-01, RX claim data with T2D product flag
2. Combine these two data sources to generate T2D flag for each patient
3. Save table to databricks

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "../00_cohort_config"

##   Identify T2D patients from Diagnosis and RX data

In [0]:
# changed hist table

query = """
    SELECT  DISTINCT PATIENT_ID
            FROM (
          SELECT DX.CLAIM_ID, DX.PATIENT_ID,DX.DIAGNOSIS_CODE,DX.PROVIDER_RENDERING_ID
          FROM PHCDW.PHCDW_DISCOVERY.STG_IQV_LAAD_DX DX
          INNER JOIN 
          ( SELECT  CLAIM_ID, MAX(LAST_DATE) AS LAST_DATE
                    FROM PHCDW.PHCDW_DISCOVERY.STG_IQV_LAAD_DX
                    GROUP BY claim_id) MAX_DX
          ON (DX.CLAIM_ID = MAX_DX.CLAIM_ID AND DX.LAST_DATE = MAX_DX.LAST_DATE)
          WHERE SERVICE_DATE <= '2018-05-31'

          UNION ALL

          SELECT CLAIM_ID, TO_VARCHAR(PATIENT_ID) AS PATIENT_ID, DIAGNOSIS_CODE, TO_VARCHAR(PROVIDER_RENDERING_ID) AS PROVIDER_RENDERING_ID
          FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_DX

          ) DX    
                  
          Join          
            
          (SELECT DISTINCT DIAGNOSIS_CODE,DIAGNOSIS_DESCRIPTION 
          FROM PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_DIAGNOSIS
                            ) CD  
          ON DX.DIAGNOSIS_CODE=CD.DIAGNOSIS_CODE
          WHERE UPPER(DIAGNOSIS_DESCRIPTION) LIKE '%TYPE 2 DIABETES%'
            AND PROVIDER_RENDERING_ID IS NOT NULL
   """
#df_dx_patient = (spark.read.format(SNOWFLAKE_SOURCE_NAME).options(**sfOptions).option("query", query).load())
df_dx_patient = Get_Data_Snowflakes(query)

In [0]:
df_dx_patient.count()

In [0]:
df_product_mapping = spark.sql("select * from kerendia_pcp.diabetes_product")
df_injection_mapping = spark.sql("select * from kerendia_pcp.diabetes_product_injection_mapping")

In [0]:
query = """
        SELECT 
        DISTINCT NDC_CD, PRODUCT_NAME,GENERIC_NAME, PRODUCT_STRENGTH, PRODUCT_FORM
        FROM PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_PRODUCT
   """
df_product = Get_Data_Snowflakes(query)

In [0]:
df_product.display()

In [0]:
df_product_mapping.display()

In [0]:
df_injection_mapping.display()

In [0]:

from pyspark.sql.functions import col, when, trim

df_product_flag = (
  df_product
  .join(df_product_mapping, how="inner", on="PRODUCT_NAME")
  .join(df_injection_mapping, how="left", on="PRODUCT_FORM")
  .filter(
      col("PRODUCT_NAME").isNotNull() &
      col("PRODUCT_FORM").isNotNull() &
      (trim(col("PRODUCT_FORM")) != "") &
      col("LOGIC").isNotNull() &
      (~col("LOGIC").cast("string").isin("null", "")) & 
      col("INJECTION").isNotNull() &
      (~col("INJECTION").isin("null", ""))
  )
  .withColumn(
      "DIABETIC_FLAG",
      when(col("LOGIC").cast("int") == 1, 1).otherwise(
          when(col("INJECTION") == "No", 1).otherwise(0)
      )
  )
)



In [0]:
df_product_flag.display()

In [0]:
# UPDATE - changed rx_hist table

query = """
        SELECT  DISTINCT PATIENT_ID, NDC_CD
        FROM 
        (
        SELECT DISTINCT CLAIM_ID, TO_VARCHAR(PATIENT_ID) AS PATIENT_ID, NDC_CD, SVC_DT
        FROM    (
                SELECT PATIENT_ID, CLAIM_ID, SVC_DT, DAYS_SUPPLY, QUANTITY, REFILL_CODE, REJECT_CODE, PROVIDER_ID, NDC_CD, PAYER_PLAN_ID 
                FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX where PATIENT_ID is not null
                ) RX
        ) where PATIENT_ID is not null
   """

df_rx = Get_Data_Snowflakes(query)

In [0]:
df_rx.count()
df_rx.limit(100).display()

In [0]:
df_rx_patient = (
  df_rx
  .join(df_product_flag, how="inner", on="NDC_CD")
  .filter("DIABETIC_FLAG = 1")
  .filter(col("PATIENT_ID").isNotNull())  # <-- filters out nulls
  .select("PATIENT_ID")
  .distinct()
)

In [0]:
df_rx_patient.count()

In [0]:
df_t2d_patient = (
  df_dx_patient.union(df_rx_patient)
  .dropDuplicates()
  .withColumn("DIABETIC_FLAG", F.lit(1))
)

In [0]:
df_t2d_patient.display()

In [0]:
# VALIDATION: write to a separate path so we don't overwrite production data
df_t2d_patient.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_t2d_patient_validation")